In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image

warnings.filterwarnings("ignore")
tqdm.pandas()

In [ ]:
DEVICE = "mps" if torch.mps.is_available() else "cpu" 

DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
IMAGE_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/images")

print(DEVICE)

In [ ]:
res_net50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(DEVICE)

In [ ]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root_dir, df):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        w, h = img.size if isinstance(img, Image.Image) else (img.shape[2], img.shape[1])
        min_side = min(w, h)
        img_cropped = transforms.CenterCrop(min_side)(img)
        img_resized = transforms.Resize((256, 256))(img_cropped)
        img_tensor = transforms.ToTensor()(img_resized)
        
        return img_tensor

In [ ]:
tr_images_ds = ImageDataset(IMAGE_FOLDER, train_df)

batch_size = 32

images = DataLoader(tr_images_ds, batch_size=batch_size, shuffle=False)

In [ ]:
sureness50 = []

for batch in tqdm(images):
    with torch.no_grad():
        image_sure50 = res_net50(batch.to(DEVICE))

    sure50 = image_sure50.to('cpu').numpy()

    sureness50.extend(sure50)

In [ ]:
for i in range(1000):
    train_df[f"f{i}"] = [sureness50[j][i] for j in range(len(sureness50))]

features = [f'f{u}' for u in range(1000)]
target = 'label'

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(train_df[features], train_df[target], test_size=0.3, shuffle=True)

In [ ]:
logreg_model = LogisticRegression(n_jobs=-1, solver='newton-cholesky')

logreg_model.fit(X_tr, y_tr)

pred = logreg_model.predict(X_val)
print("Точность на валидационной выборке:", accuracy_score(pred, y_val))

In [ ]:
logreg_model.fit(train_df[features], train_df[target])

In [ ]:
ts_images_ds = ImageDataset(IMAGE_FOLDER, test_df)

In [ ]:
images_ts = DataLoader(ts_images_ds, batch_size=batch_size, shuffle=False)

In [ ]:
sureness50_ts = []


for images_ in tqdm(images_ts):
    with torch.no_grad():
        sure = res_net50(images_.to(DEVICE))

    sureness50_ts.extend(sure.to('cpu').numpy().tolist())

In [ ]:
for i in range(1000):
    test_df[f"f{i}"] = [sureness50_ts[j][i] for j in range(len(sureness50_ts))]

pred_ts = logreg_model.predict_proba(test_df[features])

In [ ]:
test_df['y_pred'] = [i[1] for i in pred_ts.tolist()]

In [ ]:
(test_df[["id", 'y_pred']]).to_csv("submission_with_resnet.csv", index=False)